# 03 — Hyperparameter Tuning (Keras v3 / PyTorch backend)

Tunes both deep learning architectures using Keras Tuner RandomSearch.

**Key improvements over previous run:**
- 50k sample (was 30k) — more signal for the LSTM to learn from
- FNN units up to 512 (was 256)
- LSTM units up to 256 (was 128); lr range widened to include higher values
- 10 trials per model (was 8); 20 epochs per trial (was 15)
- Progress printed every epoch

> **Runtime: FNN ~20 min · LSTM ~90 min**


In [1]:
# ── MUST be first cell ───────────────────────────────────────────────────
import os
os.environ["KERAS_BACKEND"] = "torch"   # Keras v3 with PyTorch backend

import numpy as np
import pandas as pd
import json, time
import keras
import keras_tuner as kt
from keras import layers

print(f"Keras  : {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")
assert keras.backend.backend() == "torch", "Restart kernel — backend must be torch"


Keras  : 3.14.1
Backend: torch


In [2]:
# ── Load preprocessed data ────────────────────────────────────────────────
X_train = np.load("data/X_train_dense.npy")
X_val   = np.load("data/X_val_dense.npy")
seq_tr  = np.load("data/seq_train.npy")
seq_va  = np.load("data/seq_val.npy")
num_tr  = np.load("data/X_train_num.npy")
num_va  = np.load("data/X_val_num.npy")
y_train = np.load("data/y_train.npy")
y_val   = np.load("data/y_val.npy")

with open("data/meta.json") as f: meta = json.load(f)
INPUT_DIM = meta["INPUT_DIM"]
MAX_WORDS = meta["MAX_WORDS"]
MAX_LEN   = meta["MAX_LEN"]

print(f"Train: {len(y_train):,} | Val: {len(y_val):,}")
print(f"FNN input dim: {INPUT_DIM:,}  |  LSTM seq len: {MAX_LEN}")


Train: 34,999 | Val: 7,501
FNN input dim: 5,005  |  LSTM seq len: 150


In [3]:
# ── Verbose tuner — prints every epoch ───────────────────────────────────
class VerboseTuner(kt.RandomSearch):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._tc = 0

    def run_trial(self, trial, *args, **kwargs):
        self._tc += 1
        n, total = self._tc, self.oracle.max_trials
        hp = trial.hyperparameters.values
        print(f"\n{'='*62}")
        print(f"TRIAL {n}/{total}  |  {hp}")
        print(f"{'='*62}")
        t0 = time.time()

        class _Log(keras.callbacks.Callback):
            def on_epoch_end(self_, epoch, logs=None):
                print(f"  ep{epoch+1:>2}  "
                      f"loss={logs.get('loss',0):.4f}  "
                      f"val_loss={logs.get('val_loss',0):.4f}  "
                      f"val_mae={logs.get('val_mae',0):.4f}  "
                      f"({time.time()-t0:.0f}s)", flush=True)

        cbs = kwargs.pop("callbacks", []) + [_Log()]
        return super().run_trial(trial, *args, callbacks=cbs, **kwargs)


In [4]:
# ════════════════════════════════════════════════════════════════════════
# MODEL A — FNN + BoW  (Week 2)
# Wider search: units up to 512, batch_norm option, 10 trials × 20 epochs
# ════════════════════════════════════════════════════════════════════════
def build_fnn(hp):
    """FNN on TF-IDF bag-of-words + numeric features (Week 2 architecture)."""
    n_layers = hp.Int("n_layers",   1, 4)
    units    = hp.Int("units",      64, 512, step=64)
    dropout  = hp.Float("dropout",  0.1, 0.5, step=0.1)
    lr       = hp.Float("lr",       5e-5, 5e-3, sampling="log")
    use_bn   = hp.Boolean("batch_norm")

    inp = keras.Input(shape=(INPUT_DIM,), name="bow_input")
    x = inp
    for _ in range(n_layers):
        x = layers.Dense(units, activation="relu")(x)
        if use_bn:
            x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="linear", name="score")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse", metrics=["mae"],
    )
    return model

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=0)

tuner_fnn = VerboseTuner(
    build_fnn,
    objective=kt.Objective("val_mae", direction="min"),
    max_trials=10, seed=42,
    directory="kt_fnn", project_name="fnn", overwrite=True,
)

print("FNN tuning — 10 trials × 20 epochs max...")
t0 = time.time()
tuner_fnn.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=256,
    callbacks=[early_stop], verbose=0,
)
print(f"\n✅ FNN tuning done in {(time.time()-t0)/60:.1f} min")
best_hp_fnn = tuner_fnn.get_best_hyperparameters(1)[0].values
print("Best FNN hparams:", best_hp_fnn)


FNN tuning — 10 trials × 20 epochs max...

TRIAL 1/10  |  {'n_layers': 3, 'units': 64, 'dropout': 0.30000000000000004, 'lr': 0.0001748722238931748, 'batch_norm': True}
  ep 1  loss=72.7220  val_loss=65.4181  val_mae=7.9356  (3s)
  ep 2  loss=67.5384  val_loss=56.0226  val_mae=7.3384  (5s)
  ep 3  loss=62.6380  val_loss=50.8700  val_mae=7.0017  (8s)
  ep 4  loss=56.3373  val_loss=48.0276  val_mae=6.8097  (10s)
  ep 5  loss=48.9864  val_loss=43.2001  val_mae=6.4502  (12s)
  ep 6  loss=40.8046  val_loss=35.8154  val_mae=5.8479  (14s)
  ep 7  loss=32.7118  val_loss=28.1036  val_mae=5.1518  (16s)
  ep 8  loss=25.1305  val_loss=20.7371  val_mae=4.3855  (18s)
  ep 9  loss=18.8179  val_loss=14.2507  val_mae=3.5785  (20s)
  ep10  loss=14.0851  val_loss=9.4857  val_mae=2.8644  (21s)
  ep11  loss=10.5126  val_loss=6.2478  val_mae=2.2615  (23s)
  ep12  loss=8.1614  val_loss=4.3433  val_mae=1.8415  (25s)
  ep13  loss=6.8552  val_loss=2.9746  val_mae=1.4788  (27s)
  ep14  loss=5.9966  val_loss=2.306

In [5]:
# ── FNN tuning summary table ──────────────────────────────────────────────
rows = []
for trial in tuner_fnn.oracle.trials.values():
    if trial.status == "COMPLETED":
        hp = trial.hyperparameters.values
        rows.append({
            "n_layers":   hp.get("n_layers"),
            "units":      hp.get("units"),
            "dropout":    round(hp.get("dropout",0), 2),
            "batch_norm": hp.get("batch_norm"),
            "lr":         f"{hp.get('lr',0):.2e}",
            "val_mae":    round(trial.score, 4),
        })
df_fnn = pd.DataFrame(rows).sort_values("val_mae").reset_index(drop=True)
df_fnn.index += 1
print("=== FNN Tuning Results ===")
print(df_fnn.to_string())
df_fnn.to_csv("data/tuning_results_fnn.csv")
with open("data/best_hparams_fnn.json","w") as f: json.dump(best_hp_fnn,f,indent=2)


=== FNN Tuning Results ===
    n_layers  units  dropout  batch_norm        lr  val_mae
1          4    256      0.3       False  3.38e-04   0.8746
2          4    448      0.1        True  1.75e-04   0.8767
3          1    320      0.2       False  2.06e-04   0.8871
4          1    192      0.5       False  3.99e-04   0.8886
5          2    320      0.1       False  4.94e-04   0.8899
6          1    256      0.4        True  1.25e-04   0.8962
7          1    384      0.4       False  3.30e-03   0.8963
8          1    256      0.1       False  1.28e-04   0.9499
9          3     64      0.3        True  1.75e-04   0.9634
10         4    320      0.5        True  7.58e-05   1.0030


In [6]:
# ════════════════════════════════════════════════════════════════════════
# MODEL B — LSTM + Learned Embeddings  (Week 4)
# Key fix: wider lr range (1e-3 to 5e-3), lstm_units up to 256,
#          return_sequences option for stacked LSTM
# ════════════════════════════════════════════════════════════════════════
def build_lstm(hp):
    """
    LSTM with learned embeddings (Week 4 architecture).
    Text: Embedding → LSTM → Dropout
    Numeric: direct side-input
    Combined: Concatenate → Dense(64) → score
    """
    embed_dim  = hp.Int("embed_dim",   64, 256, step=64)
    lstm_units = hp.Int("lstm_units",  64, 256, step=64)
    dropout    = hp.Float("dropout",   0.1, 0.4, step=0.1)
    # Wider lr range — higher lr needed for LSTM to converge in 20 epochs
    lr         = hp.Float("lr",        5e-4, 5e-3, sampling="log")

    text_input = keras.Input(shape=(MAX_LEN,), dtype="int32", name="text_input")
    x = layers.Embedding(input_dim=MAX_WORDS, output_dim=embed_dim,
                         mask_zero=True, name="embedding")(text_input)
    x = layers.LSTM(lstm_units, name="lstm")(x)
    x = layers.Dropout(dropout)(x)

    num_input = keras.Input(shape=(5,), name="numeric_input")
    combined  = layers.Concatenate()([x, num_input])
    combined  = layers.Dense(64, activation="relu")(combined)
    out       = layers.Dense(1, activation="linear", name="score")(combined)

    model = keras.Model([text_input, num_input], out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse", metrics=["mae"],
    )
    return model

early_stop_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=0)

tuner_lstm = VerboseTuner(
    build_lstm,
    objective=kt.Objective("val_mae", direction="min"),
    max_trials=10, seed=42,
    directory="kt_lstm", project_name="lstm", overwrite=True,
)

print("LSTM tuning — 10 trials × 20 epochs max...")
print("(Each trial ~4-6 min on MPS/GPU — ~60 min total)")
t0 = time.time()
tuner_lstm.search(
    [seq_tr, num_tr], y_train,
    validation_data=([seq_va, num_va], y_val),
    epochs=20, batch_size=256,
    callbacks=[early_stop_lstm], verbose=0,
)
print(f"\n✅ LSTM tuning done in {(time.time()-t0)/60:.1f} min")
best_hp_lstm = tuner_lstm.get_best_hyperparameters(1)[0].values
print("Best LSTM hparams:", best_hp_lstm)


LSTM tuning — 10 trials × 20 epochs max...
(Each trial ~4-6 min on MPS/GPU — ~60 min total)

TRIAL 1/10  |  {'embed_dim': 192, 'lstm_units': 64, 'dropout': 0.2, 'lr': 0.0009350727883249913}
  ep 1  loss=11.9595  val_loss=1.9047  val_mae=1.0334  (34s)
  ep 2  loss=1.9783  val_loss=1.4741  val_mae=0.9036  (67s)
  ep 3  loss=1.5942  val_loss=1.4032  val_mae=0.8969  (101s)
  ep 4  loss=1.3782  val_loss=1.3612  val_mae=0.8886  (135s)
  ep 5  loss=1.2418  val_loss=1.3549  val_mae=0.8737  (168s)
  ep 6  loss=1.1457  val_loss=1.3433  val_mae=0.8660  (201s)
  ep 7  loss=1.0719  val_loss=1.3650  val_mae=0.8958  (236s)
  ep 8  loss=1.0055  val_loss=1.3938  val_mae=0.8785  (270s)
  ep 9  loss=0.9457  val_loss=1.3937  val_mae=0.8789  (304s)
  ep10  loss=0.8914  val_loss=1.4115  val_mae=0.8943  (338s)

TRIAL 2/10  |  {'embed_dim': 256, 'lstm_units': 128, 'dropout': 0.30000000000000004, 'lr': 0.0005832489702191644}
  ep 1  loss=12.3073  val_loss=2.1150  val_mae=1.1082  (35s)
  ep 2  loss=2.1599  val_

In [7]:
# ── LSTM tuning summary table ─────────────────────────────────────────────
rows = []
for trial in tuner_lstm.oracle.trials.values():
    if trial.status == "COMPLETED":
        hp = trial.hyperparameters.values
        rows.append({
            "embed_dim":  hp.get("embed_dim"),
            "lstm_units": hp.get("lstm_units"),
            "dropout":    round(hp.get("dropout",0), 2),
            "lr":         f"{hp.get('lr',0):.2e}",
            "val_mae":    round(trial.score, 4),
        })
df_lstm = pd.DataFrame(rows).sort_values("val_mae").reset_index(drop=True)
df_lstm.index += 1
print("=== LSTM Tuning Results ===")
print(df_lstm.to_string())
df_lstm.to_csv("data/tuning_results_lstm.csv")
with open("data/best_hparams_lstm.json","w") as f: json.dump(best_hp_lstm,f,indent=2)


=== LSTM Tuning Results ===
    embed_dim  lstm_units  dropout        lr  val_mae
1         256         128      0.2  3.15e-03   0.8444
2         256         128      0.2  1.30e-03   0.8625
3         192          64      0.2  9.35e-04   0.8660
4         192          64      0.2  1.56e-03   0.8720
5         256         128      0.1  9.52e-04   0.8740
6         256         128      0.3  5.83e-04   0.8828
7         256          64      0.2  1.60e-03   0.8844
8          64          64      0.3  2.42e-03   0.8880
9         128          64      0.4  2.07e-03   0.9125
10        256          64      0.4  5.51e-04   0.9757


In [8]:
print("\n=== FINAL BEST HYPERPARAMETERS ===")
print(f"FNN : {best_hp_fnn}")
print(f"LSTM: {best_hp_lstm}")
print("\n✅ Saved to data/best_hparams_fnn.json and data/best_hparams_lstm.json")



=== FINAL BEST HYPERPARAMETERS ===
FNN : {'n_layers': 4, 'units': 256, 'dropout': 0.30000000000000004, 'lr': 0.00033764319636739114, 'batch_norm': False}
LSTM: {'embed_dim': 256, 'lstm_units': 128, 'dropout': 0.2, 'lr': 0.003148367158154625}

✅ Saved to data/best_hparams_fnn.json and data/best_hparams_lstm.json
